In [2]:
import pandas as pd
import glob
from typing import Tuple

In [10]:
pwd

'c:\\Users\\samee\\Documents\\CHIEAC\\src'

In [32]:
FOLDER = r"C:\Users\samee\Documents\CHIEAC\HOPE_WSDM_2022\Train"
def list_files(FOLDER):
    files = glob.glob(FOLDER+"/*.txt")
    return files

def read_files(file_path: str) -> str:
    with open(file_path) as f:
        text = f.read()

    return text

In [33]:

files = list_files(FOLDER)

In [34]:
len(files)

158

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

sentiment_model_name = "paulagarciaserrano/roberta-depression-detection"

def sentence_level_sentiment(model_name: str, text: str) -> Tuple[str, float]:
    """
    Predict the sentiment (or classification label) of a given text using a
    Hugging Face sequence classification model.

    This function loads the specified model and tokenizer from the Hugging Face Hub,
    performs inference on the input text, and returns the predicted label along with
    its confidence score.

    Args:
        model_name (str): Name or path of the Hugging Face model to load.
            Example: "paulagarciaserrano/roberta-depression-detection"
        text (str): Input sentence or text to classify.

    Returns:
        Tuple[str, float]:
            - label (str): Predicted class label (e.g., "positive", "negative", "depression").
            - score (float): Confidence score (probability) associated with the predicted label.

    Notes:
        - The model is loaded on every function call, which may be inefficient for repeated use.
          For better performance, load the model once and reuse it.
        - Output labels depend on the model's `id2label` configuration.
        - Uses softmax over logits to compute class probabilities.

    Raises:
        OSError: If the model or tokenizer cannot be loaded.
        RuntimeError: If inference fails due to device or tensor issues.
    """
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    model.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    confidence, predicted_class_id = torch.max(probs, dim=-1)

    label = model.config.id2label[predicted_class_id.item()]
    score = confidence.item()

    return label, score
    




In [ ]:
sentence_level_sentiment(sentiment_model_name,"I feel like killing myself!!!!")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10585.43it/s]
RobertaForSequenceClassification LOAD REPORT from: paulagarciaserrano/roberta-depression-detection
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


('not depression', 0.9899266958236694)

In [25]:
emotion_model_name = "cirimus/modernbert-base-go-emotions"
def sentence_level_emotions(model_name: str, text: str, threshold: float = 0.5):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.sigmoid(outputs.logits)[0]

    emotion_scores = {
        model.config.id2label[i]: probs[i].item()
        for i in range(len(probs))
    }

    active_emotions = [
        label for label, score in emotion_scores.items()
        if score >= threshold
    ]

    return emotion_scores, active_emotions

In [29]:
emotion_scores, active_emotions = sentence_level_emotions(emotion_model_name, "I am done with this life, nothing seems to be working. I feel like killing myself")

Loading weights: 100%|██████████| 138/138 [00:00<00:00, 4928.47it/s]


In [30]:
active_emotions

['disappointment', 'sadness']

In [18]:
emotion_scores

{'admiration': 0.12406285107135773,
 'amusement': 0.0029394314624369144,
 'anger': 0.0008885187562555075,
 'annoyance': 0.003618694143369794,
 'approval': 0.1503061056137085,
 'caring': 0.020584864541888237,
 'confusion': 0.0008835129556246102,
 'curiosity': 0.0007568722940050066,
 'desire': 0.006082700565457344,
 'disappointment': 0.0025242140982300043,
 'disapproval': 0.0012921539600938559,
 'disgust': 0.00106920232065022,
 'embarrassment': 0.001531186862848699,
 'excitement': 0.14138126373291016,
 'fear': 0.004733560141175985,
 'gratitude': 0.0029000190552324057,
 'grief': 0.001609294442459941,
 'joy': 0.16597531735897064,
 'love': 0.0103138517588377,
 'nervousness': 0.005878069903701544,
 'optimism': 0.017486579716205597,
 'pride': 0.030110593885183334,
 'realization': 0.03628503903746605,
 'relief': 0.02585391141474247,
 'remorse': 0.0009241539519280195,
 'sadness': 0.003772301133722067,
 'surprise': 0.006200989242643118,
 'neutral': 0.19719091057777405}

#Domain -  Adaptation

In [ ]:
[i for i in read_files(files[100]).split("\n") if i!='']